АЛХАМУУД
1. Шинээр дата татаад шууд датабазад хадгалах, ингэснээр ID үүснэ
2. RESISTANCE and SUPPORT тодорхойлж датабаз руу ID-аар нь шүүд хадгална.

In [8]:
from utils.get_data import fetchCryptoData
import os
import time
import sys
from tabulate import tabulate
import ipywidgets as widgets
from IPython.display import display

In [ ]:
lookback = widgets.IntSlider(
    value=30,
    min=10,
    max=900,
    step=1,
    description='Test:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

symbol = 'ETHUSDT'
timePeriod = '1h'
lookback = 90000000
df = fetchCryptoData(symbol, timePeriod, lookback)

widgets.interact(fetchCryptoData, symbol='ETHUSDT', timePeriod='1h', lookback=(10, 900, 1))
# lookback.interact(fetchCryptoData, symbol='ETHUSDT', timePeriod='1h', lookback=(10, 900, 1))

display(lookback)

AttributeError: 'int' object has no attribute 'interact'

In [3]:
lookback.value

2

In [5]:
lookback.max

10

In [ ]:
from utils.get_data import fetchCryptoData
import os
import time
import sys
from tabulate import tabulate

symbol = 'ETHUSDT'
timePeriod = '1h'
lookback = 90000000
df = fetchCryptoData(symbol, timePeriod, lookback)
df.tail(2)

,time,open,high,low,close,volume
998,2025-02-18 04:00:00,2712.07,2718.09,2691.59,2701.89,13419.1059
999,2025-02-18 05:00:00,2701.89,2703.19,2701.51,2702.99,81.0940


In [19]:
from utils.apply_technicals import apply_technicals
from utils.candle_patterns import detect_engulfing_pattern
from utils.price_channels import find_extremum
from utils.price_channels import support_resistance_range
from utils.trade_analyze import trade_analyze
from utils.boilinger_band import boilinger_band_check
from utils.save2mysql import create_database_if_not_exists
from utils.save2mysql import update_db
import pandas as pd
import numpy as np

# Create a list to collect processed rows
analyzed_df = pd.DataFrame(columns=['time', 'open', 'high', 'low', 'close', 'volume','support', 'resistance','macd','macd_signal','macd_hist','rsi','stochastic-K','stochastic-D','ema','bb_middle','bb_std','bb_upper','bb_lower','bb_trend','bb_signal','near_support','near_resistance','engulfing','reversal','doji','macd_trade','rsi_trade','stochastic_trade','channel_trade','near_bb_support','near_bb_resistance', 'trade'])
# Add the necessary columns if they do not exist
if 'support' not in df.columns:
    df['support'] = np.nan
if 'resistance' not in df.columns:
    df['resistance'] = np.nan
if 'macd' not in df.columns:
    df['macd'] = np.nan
if 'macd_signal' not in df.columns:
    df['macd_signal'] = np.nan
if 'macd_hist' not in df.columns:
    df['macd_hist'] = np.nan
if 'rsi' not in df.columns:
    df['rsi'] = np.nan
if 'stochastic-K' not in df.columns:
    df['stochastic-K'] = np.nan
if 'stochastic-D' not in df.columns:
    df['stochastic-D'] = np.nan
if 'ema' not in df.columns:
    df['ema'] = np.nan
if 'bb_middle' not in df.columns:
    df['bb_middle'] = np.nan
if 'bb_std' not in df.columns:
    df['bb_std'] = np.nan
if 'bb_upper' not in df.columns:
    df['bb_upper'] = np.nan
if 'bb_lower' not in df.columns:
    df['bb_lower'] = np.nan
if 'bb_trend' not in df.columns:
    df['bb_trend'] = np.nan
if 'bb_signal' not in df.columns:
    df['bb_signal'] = np.nan
if 'near_support' not in df.columns:
    df['near_support'] = np.nan
if 'near_resistance' not in df.columns:
    df['near_resistance'] = np.nan
if 'engulfing' not in df.columns:
    df['engulfing'] = 'no'
if 'reversal' not in df.columns:
    df['reversal'] = 'no'
if 'doji' not in df.columns:
    df['doji'] = 'no'
if 'macd_trade' not in df.columns:
    df['macd_trade'] = 'wait'
if 'rsi_trade' not in df.columns:
    df['rsi_trade'] = 'wait'
if 'stochastic_trade' not in df.columns:
    df['stochastic_trade'] = 'wait'
if 'channel_trade' not in df.columns:
    df['channel_trade'] = 'wait'
if 'near_bb_support' not in df.columns:
    df['near_bb_support'] = np.nan
if 'near_bb_resistance' not in df.columns:
    df['near_bb_resistance'] = np.nan
if 'trade' not in df.columns:
    df['trade'] = 'wait'

for index, row in df.iterrows():
    # for index, row in df.iterrows(): нь historical data-г нэг нэгээр авч байгаа simulation
    # Эндээс эхлээд анализ хийж шалгах
    # ХАМГИЙН СҮҮЛЧИЙН 20 МЭДЭЭЛЭЛ -> analyzed_df = df.tail(20)
    analyzed_df = df.iloc[0:index+1].copy()
    # analyzed_df = analyzed_df.tail(20)
    # print("analyzed data =========> ",analyzed_df)

    if len(analyzed_df) > 4: # call support_resistance_range after find_extremum
        # analyzed_df = support_resistance_range(analyzed_df, 10)
        analyzed_df = detect_engulfing_pattern(analyzed_df)
    if len(analyzed_df) > 20:
        analyzed_df = apply_technicals(analyzed_df)
        print("\nAnalyzed data after technicals:")
        print("\nMACD Data:")
        print(tabulate(analyzed_df[['time', 'engulfing','macd_hist', 'macd_trade', 'rsi_trade', 'stochastic_trade', 'channel_trade', 'bb_trend', 'bb_signal']].tail(4), headers='keys', tablefmt='psql', floatfmt='.4f'))
    # print(tabulate(analyzed_df.tail(), headers='keys', tablefmt='psql', floatfmt='.4f'))

    create_database_if_not_exists()
    print("Database created")
    # update_db(row)
    latest_row = analyzed_df.iloc[-1]

    update_db(latest_row)
    # time.sleep(2)
    print("Chart saved to crypto_chart_app/chart.png")

    # time.sleep(2)
print("Analysis completed")
print(analyzed_df.tail(10))

Database created
Data updated for time: 2025-01-07 14:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 15:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 16:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 17:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 18:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 19:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 20:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 21:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 22:00:00
Chart saved to crypto_chart_app/chart.png
Database created
Data updated for time: 2025-01-07 23:00:00
Chart saved to crypto_

In [ ]:
# Boilinger band
        df_tech['ema'] = df_tech['close'].ewm(span=14, adjust=False).mean()
        df_tech = boilinger_band_check(df_tech, 20, 2, 10) # boilinger_band_check(df, window=20, num_std_dev=2, price_range=5):
        # Fill NaN values with previous values
        df_tech = df_tech.ffill()

In [ ]:
    if len(analyzed_df) > 50:
        channels = find_extremum(analyzed_df, 20)

In [ ]:
# Trade analysis
 analyzed_df = trade_analyze(analyzed_df)

In [ ]:
fds

In [14]:
%reset -f

In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import mplfinance as mpf
import ta
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from datetime import datetime

# Database credentials
db_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Tamir4578',
    'database': 'portalblog_dev'
}

table_name = 'analyzed_data'
timestamp_col = 'time'
open_col = 'open'
high_col = 'high'
low_col = 'low'
close_col = 'close'

# Create database connection string
db_connection_str = 'mysql+pymysql://{user}:{password}@{host}/{database}'.format(**db_config)
db_connection = create_engine(db_connection_str)
def fetch_data_from_db(window_size=500, start_date=None, end_date=None):
    query = f'SELECT * FROM (SELECT * FROM {table_name} ORDER BY {timestamp_col} DESC LIMIT {window_size}) AS recent_data'
    if start_date and end_date:
        query = f'SELECT * FROM (SELECT * FROM {table_name} WHERE {timestamp_col} BETWEEN "{start_date}" AND "{end_date}" ORDER BY {timestamp_col} DESC LIMIT {window_size}) AS recent_data'
    query += f' ORDER BY {timestamp_col} ASC'
    df = pd.read_sql(query, db_connection)
    # Convert the 'time' column to datetime
    df['time'] = pd.to_datetime(df['time'])

    # Set the 'time' column as the index
    df.set_index('time', inplace=True)
    return df
# df = pd.read_sql(query, db_connection)
df = fetch_data_from_db()
df.head()

,id,open,high,low,close,volume,resistance,support,macd,macd_signal,...,rsi_trade,stochastic_trade,channel_trade,bb_upper,bb_middle,bb_lower,bb_trend,bb_signal,near_bb_support,near_bb_resistance
time,,,,,,,,,,,,,,,,,,,,,
2025-01-21 14:00:00,492,3309.69,3335.44,3279.09,3289.41,46772.6,None,None,None,None,...,wait,wait,wait,3345.15,3278.17,3211.20,sideways,wait,NaN,3335.44
2025-01-21 15:00:00,493,3289.42,3302.09,3265.11,3295.10,27063.4,None,None,None,None,...,wait,wait,wait,3335.94,3275.75,3215.56,sideways,wait,NaN,NaN
2025-01-21 16:00:00,494,3295.10,3349.17,3290.50,3313.85,34589.3,None,None,None,None,...,wait,wait,wait,3334.69,3275.43,3216.18,sideways,wait,NaN,NaN
2025-01-21 17:00:00,495,3313.85,3356.59,3307.81,3351.00,20466.4,None,None,None,None,...,wait,wait,wait,3346.92,3278.74,3210.55,sideways,wait,NaN,3356.59
2025-01-21 18:00:00,496,3351.00,3368.00,3330.74,3345.76,21973.6,None,None,None,None,...,wait,sell,wait,3352.58,3280.10,3207.62,sideways,wait,NaN,NaN


# Testing tactic for combination of BB and OSCI.

In [ ]:
df['osci_bb_buy'] = df.apply(lambda row: row['close'] if pd.notnull(row['near_bb_support']) and row['stochastic_trade'] == 'buy' else None, axis=1)
df['osci_bb_sell'] = df.apply(lambda row: row['close'] if pd.notnull(row['near_bb_resistance']) and row['stochastic_trade'] == 'sell' else None, axis=1)
osci_bb_buy_count = df['osci_bb_buy'].count()
osci_bb_sell_count = df['osci_bb_sell'].count()

print(f"osci_bb_buy count: {osci_bb_buy_count}")
print(f"osci_bb_sell count: {osci_bb_sell_count}")

# Define the Bollinger Bands
bbands = mpf.make_addplot(df[['bb_upper', 'bb_middle', 'bb_lower']])

# Define the buy and sell signals
buy_signals = mpf.make_addplot(df['osci_bb_buy'], type='scatter', markersize=100, marker='*', color='red')
sell_signals = mpf.make_addplot(df['osci_bb_sell'], type='scatter', markersize=100, marker='*', color='green')

# Plot the candlestick chart with Bollinger Bands and buy/sell signals
mpf.plot(df, type='candle', addplot=[bbands, buy_signals, sell_signals], volume=True, style='yahoo', figratio=(3, 1))

# Testing the combination of BB and RSI

In [6]:
# import numpy as np

# ============ calculate slope of bb_lower ============
def bb_lower_slope(df):
    # Select the last 4 points of data
    last_points = df['bb_lower'].tail(4)
    # Calculate the slope of the 'bb_lower' line
    x = np.arange(len(last_points))
    y = last_points.values
    slope, _ = np.polyfit(x, y, 1)
    # Convert slope to degrees
    slope_in_degrees = np.degrees(np.arctan(slope))
    return slope, slope_in_degrees

slope, degree = bb_lower_slope(df)
degree
slope

2.7379999999997615

Test with trend reversal detection by candlestick

SUPPORT and RESISTANCE

In [10]:
import pandas as pd
# df = pd.read_csv("EURUSD_Candlestick_4_Hour_ASK_05.05.2003-16.10.2021.csv")
# df.columns=['time', 'open', 'high', 'low', 'close', 'volume']
#Check if NA values are in data
df=df[df['volume']!=0]
df.reset_index(drop=True, inplace=True)
df.isna().sum()
df.head(10)

,id,open,high,low,close,volume,resistance,support,macd,macd_signal,...,bb_middle,bb_lower,bb_trend,bb_signal,near_bb_support,near_bb_resistance,ath,atl,osci_bb_buy,osci_bb_sell
0,492,3309.69,3335.44,3279.09,3289.41,46772.60,None,None,None,None,...,3278.17,3211.20,sideways,wait,NaN,3335.44,NaN,NaN,NaN,NaN
1,493,3289.42,3302.09,3265.11,3295.10,27063.40,None,None,None,None,...,3275.75,3215.56,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
2,494,3295.10,3349.17,3290.50,3313.85,34589.30,None,None,None,None,...,3275.43,3216.18,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
3,495,3313.85,3356.59,3307.81,3351.00,20466.40,None,None,None,None,...,3278.74,3210.55,sideways,wait,NaN,3356.59,NaN,NaN,NaN,NaN
4,496,3351.00,3368.00,3330.74,3345.76,21973.60,None,None,None,None,...,3280.10,3207.62,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
5,497,3345.75,3346.63,3318.00,3332.16,14995.40,None,None,None,None,...,3282.51,3206.37,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
6,498,3332.16,3339.49,3304.01,3314.39,11131.50,None,None,None,None,...,3286.88,3214.23,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
7,499,3314.39,3336.50,3302.03,3331.90,12215.40,None,None,None,None,...,3290.95,3217.74,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
8,500,3331.90,3347.12,3326.70,3333.74,8771.23,None,None,None,None,...,3294.74,3220.88,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN
9,501,3333.75,3334.89,3308.11,3327.54,15981.20,None,None,None,None,...,3297.98,3224.37,sideways,wait,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
backcandles= 50
wind = 5

candleid = 400

maxim = np.array([])
minim = np.array([])
xxmin = np.array([])
xxmax = np.array([])
for i in range(candleid-backcandles, candleid+1, wind):
    minim = np.append(minim, df.low.iloc[i:i+wind].min())
    xxmin = np.append(xxmin, df.low.iloc[i:i+wind].idxmin())
for i in range(candleid-backcandles, candleid+1, wind):
    maxim = np.append(maxim, df.high.loc[i:i+wind].max())
    xxmax = np.append(xxmax, df.high.iloc[i:i+wind].idxmax())
slmin, intercmin = np.polyfit(xxmin, minim,1)
slmax, intercmax = np.polyfit(xxmax, maxim,1)

dfpl = df[candleid-backcandles:candleid+backcandles]
fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['open'],
                high=dfpl['high'],
                low=dfpl['low'],
                close=dfpl['close'])])
fig.add_trace(go.Scatter(x=xxmin, y=slmin*xxmin + intercmin, mode='lines', name='min slope'))
fig.add_trace(go.Scatter(x=xxmax, y=slmax*xxmax + intercmax, mode='lines', name='max slope'))

In [12]:
xxmin

array([351., 355., 364., 365., 371., 375., 384., 388., 392., 395., 400.])

In [13]:
minim

array([2709.2 , 2746.85, 2715.27, 2716.54, 2755.69, 2820.25, 2737.62,
       2672.03, 2655.28, 2687.  , 2668.03])

In [15]:
y=slmin*xxmin + intercmin
y

array([2751.75327056, 2746.22601642, 2733.78969461, 2732.40788108,
       2724.11699987, 2718.58974573, 2706.15342392, 2700.62616978,
       2695.09891564, 2690.95347503, 2684.04440736])

write function that save or update values of "y=slmin*xxmin + intercmin" into database by filtering 'ID'

In [19]:
def save_or_update_y_values(df, xxmin, y, table_name, db_connection):
    # Create a DataFrame with 'ID' and 'y' values
    y_df = pd.DataFrame({'ID': df.index[xxmin.astype(int)], 'y': y})

    # Iterate through the DataFrame and update or insert values into the database
    with db_connection.connect() as connection:
        for index, row in y_df.iterrows():
            query = f"""
            INSERT INTO {table_name} (ID, y)
            VALUES ({row['ID']}, {row['y']})
            ON DUPLICATE KEY UPDATE y = {row['y']}
            """
            connection.execute(query)

# Call the function
save_or_update_y_values(df, xxmin, y, 'analyzed_data', db_connection)

ObjectNotExecutableError: Not an executable object: '\n            INSERT INTO analyzed_data (ID, y)\n            VALUES (351.0, 2751.7532705626277)\n            ON DUPLICATE KEY UPDATE y = 2751.7532705626277\n            '

In [6]:
dfpl = df[candleid-wind-backcandles:candleid+backcandles]
import plotly.graph_objects as go
from datetime import datetime

fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['open'],
                high=dfpl['high'],
                low=dfpl['low'],
                close=dfpl['close'])])

#-------------------------------------------------------------------------
# Fitting intercepts to meet highest or lowest candle point in time slice
adjintercmin = df.low.loc[candleid-backcandles:candleid].min() - slmin*df.low.iloc[candleid-backcandles:candleid].idxmin()
adjintercmax = df.high.loc[candleid-backcandles:candleid].max() - slmax*df.high.iloc[candleid-backcandles:candleid].idxmax()
fig.add_trace(go.Scatter(x=xxmin, y=slmin*xxmin + adjintercmin, mode='lines', name='min slope'))
fig.add_trace(go.Scatter(x=xxmax, y=slmax*xxmax + adjintercmax, mode='lines', name='max slope'))
fig.show()

In [7]:
dfpl = df[candleid-wind-backcandles:candleid+backcandles]
import plotly.graph_objects as go
from datetime import datetime

fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['open'],
                high=dfpl['high'],
                low=dfpl['low'],
                close=dfpl['close'])])

#-------------------------------------------------------------------------
# Fitting intercepts to wrap price candles
adjintercmax = (df.high.iloc[xxmax] - slmax*xxmax).max()
adjintercmin = (df.low.iloc[xxmin] - slmin*xxmin).min()
fig.add_trace(go.Scatter(x=xxmin, y=slmin*xxmin + adjintercmin, mode='lines', name='min slope'))
fig.add_trace(go.Scatter(x=xxmax, y=slmax*xxmax + adjintercmax, mode='lines', name='max slope'))
fig.show()

backcandles time window more dynamic

In [14]:
import numpy as np
from matplotlib import pyplot
backcandles= 40 # 6*8
brange = 10 # backcandles//4 #should be less than backcandles
wind = 6

candleid = 420

optbackcandles= backcandles
sldiff = 500

for r1 in range(backcandles-brange, backcandles+brange):
    maxim = np.array([])
    minim = np.array([])
    xxmin = np.array([])
    xxmax = np.array([])
    for i in range(candleid-r1, candleid+1, wind):
        minim = np.append(minim, df.low.iloc[i:i+wind].min())
        xxmin = np.append(xxmin, df.low.iloc[i:i+wind].idxmin())
    for i in range(candleid-r1, candleid+1, wind):
        maxim = np.append(maxim, df.high.loc[i:i+wind].max())
        xxmax = np.append(xxmax, df.high.iloc[i:i+wind].idxmax())
    slmin, intercmin = np.polyfit(xxmin, minim,1)
    slmax, intercmax = np.polyfit(xxmax, maxim,1)

    if(abs(slmin-slmax)<sldiff):
        sldiff = abs(slmin-slmax)
        optbackcandles=r1
        slminopt = slmin
        slmaxopt = slmax
        intercminopt = intercmin
        intercmaxopt = intercmax
        maximopt = maxim.copy()
        minimopt = minim.copy()
        xxminopt = xxmin.copy()
        xxmaxopt = xxmax.copy()

        print(optbackcandles)
dfpl = df[candleid-wind-optbackcandles-backcandles:candleid+optbackcandles]
fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['open'],
                high=dfpl['high'],
                low=dfpl['low'],
                close=dfpl['close'])])

adjintercmax = (df.high.iloc[xxmaxopt] - slmaxopt*xxmaxopt).max()
adjintercmin = (df.low.iloc[xxminopt] - slminopt*xxminopt).min()
fig.add_trace(go.Scatter(x=xxminopt, y=slminopt*xxminopt + adjintercmin, mode='lines', name='min slope'))
fig.add_trace(go.Scatter(x=xxmaxopt, y=slmaxopt*xxmaxopt + adjintercmax, mode='lines', name='max slope'))
fig.show()

30
37
38
43


In [11]:
import numpy as np
from matplotlib import pyplot
backcandles= 100
brange = 50 #should be less than backcandles
wind = 5

candleid = 480

optbackcandles= backcandles
sldiff = 100
sldist = 500
for r1 in range(backcandles-brange, backcandles+brange):
    maxim = np.array([])
    minim = np.array([])
    xxmin = np.array([])
    xxmax = np.array([])

    for i in range(candleid-r1, candleid+1, wind):
        minim = np.append(minim, df.low.iloc[i:i+wind].min())
        xxmin = np.append(xxmin, df.low.iloc[i:i+wind].idxmin())
    for i in range(candleid-r1, candleid+1, wind):
        maxim = np.append(maxim, df.high.loc[i:i+wind].max())
        xxmax = np.append(xxmax, df.high.iloc[i:i+wind].idxmax())
    slmin, intercmin = np.polyfit(xxmin, minim,1)
    slmax, intercmax = np.polyfit(xxmax, maxim,1)

    dist = (slmax*candleid + intercmax)-(slmin*candleid + intercmin)
    if(dist<sldist): #abs(slmin-slmax)<sldiff and
        #sldiff = abs(slmin-slmax)
        sldist = dist
        optbackcandles=r1
        slminopt = slmin
        slmaxopt = slmax
        intercminopt = intercmin
        intercmaxopt = intercmax
        maximopt = maxim.copy()
        minimopt = minim.copy()
        xxminopt = xxmin.copy()
        xxmaxopt = xxmax.copy()


print(optbackcandles)
dfpl = df[candleid-wind-optbackcandles-backcandles:candleid+optbackcandles]
fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['open'],
                high=dfpl['high'],
                low=dfpl['low'],
                close=dfpl['close'])])

adjintercmax = (df.high.iloc[xxmaxopt] - slmaxopt*xxmaxopt).max()
adjintercmin = (df.low.iloc[xxminopt] - slminopt*xxminopt).min()
fig.add_trace(go.Scatter(x=xxminopt, y=slminopt*xxminopt + adjintercmin, mode='lines', name='min slope'))
fig.add_trace(go.Scatter(x=xxmaxopt, y=slmaxopt*xxmaxopt + adjintercmax, mode='lines', name='max slope'))
fig.show()

149
